---------------------------------------------------------------------------------------------------------------
# Summary PDF Generation and Saving
---------------------------------------------------------------------------------------------------------------
## Content Log
1. Import Libraries
2. Load Data
3. Creating Overall Summary and Merging
4. Filtering Top 10 Volatile Stocks
5. Defining the Path and the structure of document
6. Structuring the PDF 
7. Appending the heading and the data to the PDF
8. Creating Intro and the Para of Document
9. Creatinga and Styling Heading for top Volatile Stocks and adding
10. Creating and Styling Multi-level Insights
11. Sample Stock Price trend
12. Creating and Adding Key insights of document
13. Creating and Saving the Finaly PDF report
---------------------------------------------------------------------------------------------------------------

# 01. Import Libraries

In [1]:
import os
import pandas as pd
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Image,
    Table, TableStyle, PageBreak
)
from reportlab.lib import colors
from reportlab.lib.pagesizes import letter
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.units import mm

# 02. Load Data

In [2]:
data_path = "../../data/raw_data"

datasets = {}
for file in os.listdir(data_path):
    if file.endswith(".csv"):
        df = pd.read_csv(os.path.join(data_path, file), low_memory=False)
        df["Ticker"] = file.replace(".csv", "")
        datasets[file] = df

print(f"Total datasets loaded: {len(datasets)}")

Total datasets loaded: 30


# 03. Creating Overall Summary and Merging

In [3]:
summary_list = []
market_summary = []
movement_summary = []
volatility_summary = []

for name, df in datasets.items():
    summary_list.append({
        "Dataset": name,
        "Rows": df.shape[0],
        "Columns": df.shape[1]
    })

    if "Date" in df.columns:
        df["Date"] = pd.to_datetime(df["Date"])
        df.sort_values("Date", inplace=True)

    if "Close" in df.columns:
        market_summary.append({
            "Dataset": name,
            "Mean_Close": df["Close"].mean(),
            "Std_Close": df["Close"].std()
        })

        df["Next_Close"] = df["Close"].shift(-1)
        df["Movement"] = (df["Next_Close"] > df["Close"]).astype(int)

        movement_summary.append({
            "Dataset": name,
            "UP_Ratio": df["Movement"].mean()
        })

        volatility_summary.append({
            "Dataset": name,
            "Volatility": df["Close"].pct_change().std()
        })

summary_df = pd.DataFrame(summary_list)
market_df = pd.DataFrame(market_summary)
movement_df = pd.DataFrame(movement_summary)
volatility_df = pd.DataFrame(volatility_summary)

final_summary = (
    summary_df
    .merge(market_df, on="Dataset", how="left")
    .merge(movement_df, on="Dataset", how="left")
    .merge(volatility_df, on="Dataset", how="left")
)

# 04. Filtering Top 10 Volatile Stocks

In [4]:
top_volatile = final_summary.sort_values(by="Volatility", ascending=False).head(10)

# 05. Defining the Path and the structure of document

In [5]:
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), "../../"))

PLOTS_DIR = os.path.join(BASE_DIR, "visuals_and_reports", "eda_plots")
DOCS_DIR = os.path.join(BASE_DIR, "docs", "observations")

output_path = os.path.join(DOCS_DIR, "eda_summary_report.pdf")

os.makedirs(DOCS_DIR, exist_ok=True)

key_images = {
    "Multi-Ticker Price Trend": os.path.join(PLOTS_DIR, "multi_ticker_combined_price_trends.png"),
    "Cross-Stock Correlation Heatmap": os.path.join(PLOTS_DIR, "cross_stock_correlationheatmap.png"),
    "Risk vs Return Analysis": os.path.join(PLOTS_DIR, "risk_vs_return_plot.png")
}

price_plot_dir = os.path.join(PLOTS_DIR, "price_over_time_lineplots")

# 06. Structuring the PDF 

In [6]:
from reportlab.lib.styles import ParagraphStyle
from reportlab.lib.enums import TA_CENTER

doc = SimpleDocTemplate(output_path, pagesize=letter)
styles = getSampleStyleSheet()


title_style = ParagraphStyle(
    name="TitleStyle",
    parent=styles["Title"],
    alignment=TA_CENTER,
    spaceAfter=20
)

heading_style = ParagraphStyle(
    name="HeadingStyle",
    parent=styles["Heading2"],
    spaceBefore=10,
    spaceAfter=10
)

subheading_style = ParagraphStyle(
    name="SubHeadingStyle",
    parent=styles["Heading3"],
    spaceAfter=8
)

body_style = ParagraphStyle(
    name="BodyStyle",
    parent=styles["Normal"],
    spaceAfter=10,
    leading=14
)

elements = []

# 07. Appending the heading and the data to the PDF

In [7]:
elements.append(Paragraph("Financial Data EDA Report", title_style))

elements.append(Paragraph(
    "Automated Financial MLOps Platform<br/>"
    "Author: Bhawesh Sinha<br/>"
    "Analysis Type: Exploratory Data Analysis (EDA)",
    body_style
))

elements.append(Spacer(1, 20))

# 08. Creating Intro and the Para of Document

In [8]:
elements.append(Paragraph("1. Introduction", heading_style))

elements.append(Paragraph(
    "This report presents a structured exploratory analysis of financial datasets. "
    "The objective is to understand stock behavior and support machine learning models "
    "for predicting next-day price movement (UP/DOWN).",
    body_style
))

# 09. Creatinga and Styling Heading for top Volatile Stocks and adding

In [9]:
elements.append(Paragraph("2. Top 10 Most Volatile Stocks", heading_style))

table_data = [top_volatile.columns.tolist()] + top_volatile.round(4).values.tolist()

table = Table(table_data, repeatRows=1)

table.setStyle(TableStyle([
    ("BACKGROUND", (0, 0), (-1, 0), colors.darkblue),
    ("TEXTCOLOR", (0, 0), (-1, 0), colors.whitesmoke),
    ("ALIGN", (0, 0), (-1, -1), "CENTER"),
    ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
    ("BOTTOMPADDING", (0, 0), (-1, 0), 8),
    ("GRID", (0, 0), (-1, -1), 0.5, colors.grey),
]))

elements.append(table)
elements.append(Spacer(1, 20))
elements.append(PageBreak())

# 10. Creating and Styling Multi-level Insights

In [10]:
elements.append(Paragraph("3. Market-Level Insights", heading_style))

for title, path in key_images.items():
    if os.path.exists(path):
        elements.append(Paragraph(title, subheading_style))
        
        img = Image(path, width=480, height=280)
        img.hAlign = "CENTER" 
        
        elements.append(img)
        elements.append(Spacer(1, 20))

elements.append(PageBreak())

# 11. Sample Stock Price trend

In [11]:
elements.append(Paragraph("4. Sample Stock Price Trends", heading_style))

if os.path.exists(price_plot_dir):
    for img_file in sorted(os.listdir(price_plot_dir))[:5]:
        img_path = os.path.join(price_plot_dir, img_file)

        elements.append(Paragraph(img_file.replace(".png", ""), body_style))
        
        img = Image(img_path, width=450, height=250)
        img.hAlign = "CENTER"
        
        elements.append(img)
        elements.append(Spacer(1, 15))

# 12. Creating and Adding Key insights of document

In [12]:
elements.append(PageBreak())

elements.append(Paragraph("5. Key Insights & Interpretation", heading_style))

insights = """
• High volatility stocks indicate higher prediction uncertainty.<br/>
• UP ratio near 0.5 suggests near-random movement patterns.<br/>
• Low volatility stocks are more stable and easier to model.<br/>
• Correlation across stocks indicates sector-level dependencies.<br/>
• Risk vs Return highlights trade-offs critical for financial decision-making.<br/><br/>

These insights guide feature engineering, model selection, and evaluation in the MLOps pipeline.
"""

elements.append(Paragraph(insights, body_style))

# 13. Creating and Saving the Finaly PDF report

In [13]:
def add_page_number(canvas, doc):
    page_num_text = f"Page {doc.page}"
    canvas.setFont("Helvetica", 9)
    canvas.drawRightString(200*mm, 15*mm, page_num_text)
doc.build(elements, onFirstPage=add_page_number, onLaterPages=add_page_number)